# Parte 1 – Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| | |
|---|---|
| **Fecha de extracción** | 25 de octubre de 2026 |
| **Corpus A – Documentos** | 25 imágenes de documentos peruanos de Wikimedia Commons: 8 impresos, 8 fotos con sombras, reflejos o perspectiva y 9 manuscritos |
| **Licencias (documentos)** | 15 dominio público, 2 CC0, 7 CC BY-SA (3.0 y 4.0), 1 CC BY 2.0. Detalle por archivo (URL, autor, licencia) en `datos/fuentes.csv` |
| **Corpus B – Audio** | *(se completa en la sección de ASR)* |
| **OCR** | PaddleOCR 3.7.0, modelos PP-OCRv6 *medium* (detección + reconocimiento), motor PyTorch (`engine="transformers"`) en la GPU de la Mac (MPS) |
| **ASR** | *(se completa en la sección de ASR)* |
| **Entorno** | `.venv` del repositorio del curso (`uv sync --group docai` y `uv sync --group audio`) |

Los archivos originales se guardan en `datos/raw/`, que no se versiona en Git (ver `.gitignore`).
Para recuperarlos basta con ejecutar este notebook: la sección 1 los descarga de nuevo a partir de `datos/fuentes.csv`.


In [1]:
import json
import logging
import os
import re
import time
import warnings
from pathlib import Path

# Estas dos líneas van ANTES de importar los modelos (igual que en la Sesión 10)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import paddleocr
import pandas as pd
import requests
import torch
from paddleocr import PaddleOCR
from PIL import Image, ImageDraw

for nombre in ("paddlex", "paddleocr"):
    logging.getLogger(nombre).setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 80)

DATOS = Path("datos")
RAW_DOCS = DATOS / "raw" / "documentos"
RAW_OCR = RAW_DOCS / "ocr"
RAW_OCR.mkdir(parents=True, exist_ok=True)

DISPOSITIVO = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("PaddleOCR", paddleocr.__version__, "| PyTorch", torch.__version__, "| dispositivo:", DISPOSITIVO)

PaddleOCR 3.7.0 | PyTorch 2.13.0 | dispositivo: mps


## 1 · Documentos: descarga desde Wikimedia Commons

Los 25 documentos se eligieron y revisaron a mano. Cada uno está registrado en `datos/fuentes.csv` con su
URL, autor y licencia. La descarga sigue el patrón de la Sesión 10: un `User-Agent` descriptivo,
pausas de 1,5 s entre solicitudes y no volver a bajar lo que ya está en disco.

- **Imágenes:** se piden a 1600 px de ancho (Commons entrega el tamaño estándar más cercano, entre 1200 y 1920 px).
- **PDF:** se usa la página 1, renderizada a 1280 px, que es el máximo que ofrece Commons para estos archivos.

In [2]:
API = "https://commons.wikimedia.org/w/api.php"
AGENTE = {"User-Agent": "tarea2-ds-python/2026 (https://github.com/isabellaach/python-qlab)"}


def url_imagen(titulo):
    "URL de la versión reducida de un archivo de Commons (página 1 si es un PDF)."
    params = dict(action="query", titles="File:" + titulo, prop="imageinfo", iiprop="url",
                  format="json", formatversion=2)
    if titulo.lower().endswith(".pdf"):
        params["iiurlparam"] = "page1-1280px"
    else:
        params["iiurlwidth"] = 1600
    r = requests.get(API, params=params, headers=AGENTE, timeout=60)
    r.raise_for_status()
    time.sleep(1.5)                                   # cortesía con el servidor
    return r.json()["query"]["pages"][0]["imageinfo"][0]["thumburl"]


def descargar(titulo, destino):
    "Baja un documento de Commons. Si ya está en disco, no hace nada."
    if not destino.exists():
        r = requests.get(url_imagen(titulo), headers=AGENTE, timeout=120)
        r.raise_for_status()
        destino.write_bytes(r.content)
        time.sleep(1.5)
    return destino


fuentes = pd.read_csv(DATOS / "fuentes.csv")
docs = fuentes[fuentes["modalidad"] == "documento"].copy()
docs["ruta"] = [descargar(t, Path(a)) for t, a in zip(docs["titulo"], docs["archivo"])]
docs["tamaño"] = [Image.open(r).size for r in docs["ruta"]]

print(docs["tipo"].value_counts().to_string(), "\n")
docs[["id", "tipo", "titulo", "licencia", "tamaño"]]

tipo
manuscrito    9
impreso       8
foto          8 



,id,tipo,titulo,licencia,tamaño
0,doc_001,impreso,Comunicado oficial N 21 sobre el fallecimiento de 2 personas por covid-19 en...,Public domain,"(893, 1280)"
1,doc_002,impreso,El Comercio - Lima. Perú y Ecuador. 3-3-1842.jpg,Public domain,"(675, 925)"
2,doc_003,impreso,"Carátula de El Montonero (n. 12, año 1834).jpg",Public domain,"(1430, 2239)"
3,doc_004,impreso,Inundación Carhuaz 1844.png,Public domain,"(1376, 1950)"
4,doc_005,impreso,La Tribuna-1-edicion.jpg,Public domain,"(904, 1246)"
5,doc_006,impreso,"Assassination of Antonio Miro Quesada, 1935.jpg",Public domain,"(1174, 1725)"
6,doc_007,impreso,Proclama Don Josef Fernando de Abascal y Sousa (1808).pdf,Public domain,"(1280, 1854)"
7,doc_008,impreso,"4000004161-0016 - Carta de Carlos J. Bachmann. 1914 julio 4, Maldonado; para...",CC0,"(1280, 1682)"
8,doc_009,foto,Placa del Edificio Fénix.jpg,CC BY-SA 4.0,"(1920, 1440)"
9,doc_010,foto,"N°1 Edition Of ""El Peruano"" Newspaper.jpg",CC BY-SA 4.0,"(1920, 1280)"


## 2 · OCR con PaddleOCR

Usamos la misma configuración de la clase y **no la ajustamos según los resultados**. Es la línea base de la Parte 2:

- PP-OCRv6 *medium* para detección y reconocimiento, con el motor PyTorch en la GPU de la Mac.
- Sin clasificador de orientación de página, sin "planchado" de hojas dobladas y sin orientación por línea.
- **Sin preprocesamiento:** las imágenes entran tal como vienen de Commons (color, sin binarizar ni enderezar).
  Así vemos cuánto pierde el modelo con las fotos inclinadas y los manuscritos.

Por cada documento guardamos un respaldo crudo en `datos/raw/documentos/ocr/`:
- un `.json` con cada línea leída, su confianza y su caja;
- una imagen `_cajas.jpg` con las cajas dibujadas, verdes si la confianza es alta, amarillas si es media y rojas si es baja.

Si el `.json` ya existe, no se vuelve a ejecutar el OCR.

In [3]:
OPCIONES = dict(
    use_doc_orientation_classify=False,   # no girar la página completa
    use_doc_unwarping=False,              # no "planchar" hojas dobladas
    use_textline_orientation=False,       # no girar cada línea
)


def redes(obj, _prof=0, _vistos=None):
    "Busca, dentro de un modelo de PaddleOCR, las redes neuronales de PyTorch (función de la Sesión 10)."
    _vistos = set() if _vistos is None else _vistos
    if id(obj) in _vistos or _prof > 7:
        return
    _vistos.add(id(obj))
    if isinstance(obj, torch.nn.Module):
        yield obj
        return
    for hijo in (vars(obj).values() if hasattr(obj, "__dict__") else []):
        for x in (hijo.values() if isinstance(hijo, dict) else [hijo]):
            if not isinstance(x, (str, int, float, bool, type(None), list, tuple, set)):
                yield from redes(x, _prof + 1, _vistos)


ocr = PaddleOCR(engine="transformers", **OPCIONES)
for red in redes(ocr):
    red.to(DISPOSITIVO)
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (PP-OCRv6 medium det+rec, engine=transformers, {DISPOSITIVO}, sin preprocesamiento)"
MODELO_OCR

Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

'PaddleOCR 3.7.0 (PP-OCRv6 medium det+rec, engine=transformers, mps, sin preprocesamiento)'

In [4]:
COLORES = {"alta": (12, 163, 12), "media": (250, 178, 25), "baja": (208, 59, 59)}


def nivel(confianza):
    return "alta" if confianza >= 0.9 else ("media" if confianza >= 0.7 else "baja")


def leer(ruta):
    "OCR de una imagen: devuelve una lista de líneas (texto, confianza y caja) y los segundos que tardó."
    t0 = time.perf_counter()
    r = ocr.predict(str(ruta))[0]
    segundos = time.perf_counter() - t0
    cajas = np.asarray(r["rec_boxes"]).reshape(-1, 4).tolist()
    lineas = [{"texto": t, "confianza": round(float(c), 4), "caja": [int(v) for v in b]}
              for t, c, b in zip(r["rec_texts"], r["rec_scores"], cajas)]
    return lineas, segundos


def dibujar(ruta, lineas, destino):
    "Guarda una copia de la imagen con las cajas de cada línea, coloreadas por confianza."
    im = Image.open(ruta).convert("RGB")
    d = ImageDraw.Draw(im)
    for l in lineas:
        d.rectangle(l["caja"], outline=COLORES[nivel(l["confianza"])], width=3)
    im.save(destino, quality=85)


def ocr_documento(id_, ruta):
    "Lee un documento y guarda el respaldo crudo. Si ya fue leído, reutiliza el respaldo."
    crudo = RAW_OCR / f"{id_}.json"
    if crudo.exists():
        return json.loads(crudo.read_text(encoding="utf-8"))
    lineas, segundos = leer(ruta)
    salida = {"id": id_, "archivo": str(ruta), "modelo_ocr": MODELO_OCR,
              "segundos": round(segundos, 2), "lineas": lineas}
    crudo.write_text(json.dumps(salida, ensure_ascii=False, indent=1), encoding="utf-8")
    dibujar(ruta, lineas, RAW_OCR / f"{id_}_cajas.jpg")
    return salida


leer(docs["ruta"].iloc[0])        # primera pasada de calentamiento: carga los modelos en la GPU
resultados = {id_: ocr_documento(id_, ruta) for id_, ruta in zip(docs["id"], docs["ruta"])}
print("Documentos leídos:", len(resultados))

Documentos leídos: 25


In [5]:
def resumen(r):
    conf = [l["confianza"] for l in r["lineas"]]
    texto = "\n".join(l["texto"] for l in r["lineas"])
    return {"id": r["id"], "lineas": len(conf), "caracteres": len(texto),
            "confianza_media": round(float(np.mean(conf)), 3) if conf else None,
            "lineas_baja_conf": sum(c < 0.7 for c in conf), "segundos": r["segundos"], "texto_ocr": texto}


ocr_docs = docs[["id", "tipo"]].merge(pd.DataFrame([resumen(r) for r in resultados.values()]), on="id")
print("Tiempo total de OCR:", round(ocr_docs["segundos"].sum(), 1), "s\n")
print(ocr_docs.groupby("tipo")[["lineas", "caracteres", "confianza_media", "segundos"]].mean().round(2).to_string(), "\n")
ocr_docs.drop(columns="texto_ocr")

Tiempo total de OCR: 125.0 s

            lineas  caracteres  confianza_media  segundos
tipo                                                     
foto         36.50     1238.50             0.87      4.80
impreso      98.50     3914.00             0.96      6.13
manuscrito   22.67      583.22             0.71      4.18 



,id,tipo,lineas,caracteres,confianza_media,lineas_baja_conf,segundos
0,doc_001,impreso,23,955,0.959,1,0.97
1,doc_002,impreso,121,5134,0.963,0,5.90
2,doc_003,impreso,76,2597,0.988,0,4.55
3,doc_004,impreso,136,5128,0.990,0,6.92
4,doc_005,impreso,237,7028,0.902,8,13.19
5,doc_006,impreso,146,8091,0.961,2,11.54
6,doc_007,impreso,24,944,0.980,0,2.27
7,doc_008,impreso,25,1435,0.947,1,3.71
8,doc_009,foto,15,778,0.953,1,3.51
9,doc_010,foto,96,3334,0.560,82,8.94


Un vistazo a un documento de cada tipo: el inicio del texto leído y su confianza media.

In [6]:
for tipo in ["impreso", "foto", "manuscrito"]:
    fila = ocr_docs[ocr_docs["tipo"] == tipo].iloc[0]
    print(f"── {fila['id']} ({tipo}, confianza media {fila['confianza_media']}) ──")
    print(fila["texto_ocr"][:400], "\n")

── doc_001 (impreso, confianza media 0.959) ──
PERÚ
Ministerio
Oficina General
de Salud
de Comunicaciones
COMUNICADO OFICIAL DE PRENSA N°21
El Ministerio de Salud lamenta informar el sensible fallecimiento de
dos personas por infección por COVID-19.
Se trata de un varón de 47 años con antecedentes de viaje a España,
que presentaba asma bronquial y obesidad. Fue internado en la
unidad de cuidados intensivos del hospital 2 de Mayo, por presentar 

── doc_009 (foto, confianza media 0.953) ──
EDIFICIOFENIX
EL EDIFICIO FENIX FVE CONSTRVIDO EN 1934 POR LA FIRMA FRED T. LEY
CIA. LTD. POR ENCARCO DE LA COMPAÑIA DE SEGVROS LA FENIX PERVANA
PARA ALBERGAR SV SEDE SOCIAL.
EL ENCARCO CONSISTIO EN ANADIR DOS PISOS Y VNA AZOTEA DE
DEPARTAMENTOS A VNA FINCA EXISTENTE DE VN SOLO PISO DONDE
FVNCIONABAN ALMACENES Y COMERCIOS EN RENTA. EN LA JEGVNDA PLANTA
FVNCIONARON LAS OFICINAS DE LA COMPANÍA DE SE 

── doc_017 (manuscrito, confianza media 0.658) ──
Fugill, 29 de pili 917
ot. oni lai Egut
SAannnco.
hom

## 3 · Audio: descarga del corpus

Dos fuentes, ambas con licencia CC BY-SA 4.0:

- **23 clips de Wikimedia Commons:** entrevistas en español peruano, grabadas en la calle, en ferias, en una conferencia de prensa y en fiestas patronales (2022–2024).
  Se descartaron a propósito los dos testimonios más largos (8 min cada uno) para mantener el corpus en ~53 min: sobra margen sobre el mínimo de 30 y la revisión humana de las transcripciones es más manejable.
  De los videos `.webm` se baja la versión ligera de 240p que genera Commons y se usa solo su audio.
  Uno de los clips (`El Agustino 2`) tiene dos hablantes y sirve para la diarización.
- **6 frases de OpenSLR 73** (*Crowdsourced high-quality Peruvian Spanish speech data set*, Google): lectura limpia de 3 mujeres y 3 hombres,
  una frase por hablante, elegidas al azar con semilla fija (2026). El zip pesa ~900 MB, pero se lee por partes (HTTP Range) y solo se bajan las frases elegidas;
  la clase `ZipRemoto` es de `datos.py` de la Sesión 13.

Los originales quedan en `datos/raw/audio/originales/` y cada clip se convierte a WAV mono de 16 kHz (`datos/raw/audio/clip_XXX.wav`) decodificando con PyAV.
Nada de esto se versiona: para recuperarlo basta con volver a ejecutar esta sección.

**Seudonimización:** varias descripciones de Commons mencionan el nombre de la persona entrevistada.
Esos nombres no se copian a nuestras tablas. La URL de la fuente se mantiene en `fuentes.csv` porque la licencia exige la atribución.


In [7]:
import io
import random
import urllib.parse
import urllib.request
import zipfile

import av
import soundfile as sf

import voz

RAW_AUDIO = DATOS / "raw" / "audio"
ORIGINALES = RAW_AUDIO / "originales"
ORIGINALES.mkdir(parents=True, exist_ok=True)

# (archivo en Commons, condición de grabación)
AUDIOS_COMMONS = [
    ("Entrevista 1 en la Av. Rivagüero - El Agustino.ogg", "calle"),
    ("Entrevista 2 en la Av. Rivagüero - El Agustino.ogg", "calle, 2 hablantes"),
    ("Impresiones frente al partido Perú vs Australia.opus", "calle"),
    ("Entrevista-Aurora Coronado-organización de mujeres indígenas-fiestas patrias.ogg", "conferencia de prensa"),
    ("Entrevista a la señora Aurelia, mujer shuyasina.ogg", "fiesta patronal"),
    ("Entrevista a la señora Dina Espinoza, mujer pomabambina.ogg", "fiesta patronal"),
    ("Entrevista a María Romero, organizadora de la fiesta patronal de Tayta Palli de Chuyas.ogg", "fiesta patronal"),
    ("Entrevista 2.ogg", "calle"),
    ("Entrevista a un adulto mayor.webm", "calle"),
    ("Entrevista a una vendedora de chucherias.webm", "calle"),
    ("Entrevista a un tatuador 01.webm", "calle"),
    ("Entrevista con motivo de Fiestas Patrias Perú a persona líder en el apoyo a refugios animales.webm", "calle"),
    ("Entrevista a señor presente en las manifestaciones durante las celebraciones del 28 de julio en Peru.webm", "manifestación"),
    ("Entrevista en Magdalena por Fiestas Patrias Perú 2022 (1).webm", "feria"),
    ("Entrevista en Magdalena por Fiestas Patrias Perú 2022 (3).webm", "feria"),
    ('Entrevista por Fiestas Patrias - Feria Gastronómica "SABORES Y SABERES DEL PERÚ 2022".webm', "feria"),
    ("Entrevista con motivo del 201 aniversario de la independencia del Perú.webm", "calle"),
    ("Entrevista Por el día de la Independencia del Perú.webm", "calle"),
    ("Entrevista a un ciudadano por 28 de Julio (Fiestas Patrias en el Perú).webm", "calle"),
    ("Entrevista a ciudadano del Perú.ogg", "calle"),
    ("Entrevista 1,.wav", "calle"),
    ("Entrevista 2.wav", "calle"),
    ("Entrevista 4.wav", "calle"),
]


def info_commons(titulo):
    "URL de descarga (versión ligera si es video), licencia y autor de un archivo de Commons."
    r = requests.get(API, params=dict(action="query", titles="File:" + titulo, prop="videoinfo",
                                      viprop="url|derivatives|extmetadata", format="json", formatversion=2),
                     headers=AGENTE, timeout=60)
    r.raise_for_status()
    time.sleep(1.5)
    vi = r.json()["query"]["pages"][0]["videoinfo"][0]
    ligero = [d["src"] for d in vi["derivatives"] if d.get("transcodekey") == "240p.vp9.webm"]
    url = ligero[0] if ligero else vi["url"]             # si es un video, su versión de 240p
    em = vi["extmetadata"]
    autor = re.sub(r"<[^>]+>", "", em.get("Artist", {}).get("value", "")).strip()
    return url, em.get("LicenseShortName", {}).get("value", ""), autor


def bajar(url, destino):
    if not destino.exists():
        r = requests.get(url, headers=AGENTE, timeout=300)
        r.raise_for_status()
        destino.write_bytes(r.content)
        time.sleep(1.5)
    return destino


def decodificar(ruta):
    """Cualquier audio o video -> arreglo float32 mono a 16 kHz, con PyAV (sin ffmpeg).

    No usamos voz.cargar_audio con rutas: con PyAV 19, faster_whisper.decode_audio falla
    (el pyproject del curso lo advierte) y soundfile no abre los .ogg que traen video.
    """
    remuestreo = av.AudioResampler(format="flt", layout="mono", rate=voz.SR)
    partes = []
    with av.open(str(ruta)) as contenedor:
        try:
            for cuadro in contenedor.decode(contenedor.streams.audio[0]):
                partes += [f.to_ndarray().reshape(-1) for f in remuestreo.resample(cuadro)]
        except av.error.InvalidDataError:
            pass                                   # algunos .ogg traen un último paquete dañado
    partes += [f.to_ndarray().reshape(-1) for f in remuestreo.resample(None)]
    return np.concatenate(partes).astype(np.float32)


def a_wav16k(origen, destino):
    "Guarda el clip como WAV mono de 16 kHz, la frecuencia que esperan los modelos de voz."
    if not destino.exists():
        sf.write(destino, decodificar(origen), voz.SR)
    return destino


filas_audio = []
for k, (titulo, condicion) in enumerate(AUDIOS_COMMONS, start=1):
    id_ = f"clip_{k:03d}"
    url, licencia, autor = info_commons(titulo)
    original = bajar(url, ORIGINALES / f"{id_}{'.webm' if '240p' in url else Path(url.split('?')[0]).suffix}")
    wav = a_wav16k(original, RAW_AUDIO / f"{id_}.wav")
    filas_audio.append({"id": id_, "modalidad": "audio", "fuente": "Wikimedia Commons", "condicion": condicion,
                        "fuente_url": "https://commons.wikimedia.org/wiki/File:" + urllib.parse.quote(titulo.replace(" ", "_")),
                        "licencia": licencia, "autor": autor, "titulo": titulo, "archivo": str(wav),
                        "acceso": "Descarga pública desde Wikimedia Commons" + (" (versión 240p del video; solo se usa el audio)" if "240p" in url else "")})
print("Clips de Commons:", len(filas_audio))

Clips de Commons: 23


In [8]:
OPENSLR73 = "https://www.openslr.org/resources/73/es_pe_{sexo}.zip"


def _get(url, rango=None):
    cab = dict(AGENTE)
    if rango:
        cab["Range"] = f"bytes={rango[0]}-{rango[1]}"
    with urllib.request.urlopen(urllib.request.Request(url, headers=cab), timeout=120) as r:
        return r.read()


class ZipRemoto(io.RawIOBase):
    "Archivo remoto que se lee por partes (HTTP Range): zipfile solo pide los bytes que necesita. (datos.py, Sesión 13)"

    BLOQUE = 1 << 20

    def __init__(self, url):
        self.url, self.pos, self.buf_ini, self.buf = url, 0, 0, b""
        with urllib.request.urlopen(urllib.request.Request(url, headers=AGENTE, method="HEAD"), timeout=60) as r:
            self.size = int(r.headers["Content-Length"])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        fin = self.pos + n
        if not (self.buf_ini <= self.pos and fin <= self.buf_ini + len(self.buf)):
            self.buf_ini = self.pos
            self.buf = _get(self.url, (self.pos, min(self.pos + max(n, self.BLOQUE), self.size) - 1))
        datos = self.buf[self.pos - self.buf_ini:fin - self.buf_ini]
        self.pos += len(datos)
        return datos


SEMILLA, POR_SEXO = 2026, 3
k = len(AUDIOS_COMMONS)
oficiales = []     # transcripción oficial de OpenSLR: solo como control, NO es nuestra referencia manual
for sexo in ["female", "male"]:
    z = zipfile.ZipFile(ZipRemoto(OPENSLR73.format(sexo=sexo)))
    tabla = pd.DataFrame([l.split("\t") for l in z.read("line_index.tsv").decode("utf-8").strip().splitlines()],
                         columns=["archivo_zip", "texto"])
    tabla["hablante"] = tabla["archivo_zip"].str.rsplit("_", n=1).str[0]
    tabla = tabla[tabla["texto"].str.split().str.len().between(6, 20)]
    hablantes = sorted(tabla["hablante"].unique())
    random.Random(SEMILLA).shuffle(hablantes)
    for h in hablantes[:POR_SEXO]:
        fila = tabla[tabla["hablante"] == h].sample(1, random_state=SEMILLA).iloc[0]
        k += 1
        id_ = f"clip_{k:03d}"
        original = ORIGINALES / f"{id_}.wav"
        if not original.exists():
            original.write_bytes(z.read(f"{fila['archivo_zip']}.wav"))
        wav = a_wav16k(original, RAW_AUDIO / f"{id_}.wav")
        oficiales.append({"id": id_, "archivo_zip": fila["archivo_zip"], "texto_oficial": fila["texto"].strip()})
        filas_audio.append({"id": id_, "modalidad": "audio", "fuente": "OpenSLR 73", "condicion": "lectura limpia",
                            "fuente_url": "https://www.openslr.org/73/", "licencia": "CC BY-SA 4.0", "autor": "Google",
                            "titulo": f"es_pe_{sexo}.zip / {fila['archivo_zip']}.wav", "archivo": str(wav),
                            "acceso": f"Zip es_pe_{sexo}.zip de openslr.org, archivo {fila['archivo_zip']}.wav (lectura por partes con HTTP Range)"})
pd.DataFrame(oficiales).to_csv(RAW_AUDIO / "openslr73_transcripcion_oficial.csv", index=False)
print("Frases de OpenSLR 73:", len(oficiales))

Frases de OpenSLR 73: 6


In [9]:
audios = pd.DataFrame(filas_audio)
audios["duracion_seg"] = [round(sf.info(a).duration, 1) for a in audios["archivo"]]

# fuentes.csv: se conservan las filas de documentos y se reemplazan las de audio
fuentes = pd.read_csv(DATOS / "fuentes.csv")
fuentes = pd.concat([fuentes[fuentes["modalidad"] == "documento"],
                     audios.rename(columns={"condicion": "tipo"})[["id", "modalidad", "fuente_url", "licencia", "acceso", "tipo", "archivo", "titulo", "autor"]]],
                    ignore_index=True)
fuentes.to_csv(DATOS / "fuentes.csv", index=False)

print(f"Clips: {len(audios)} | minutos: {audios['duracion_seg'].sum() / 60:.1f}\n")
print(audios.groupby("fuente")["duracion_seg"].agg(["count", "sum"]).assign(minutos=lambda d: (d["sum"] / 60).round(1)).drop(columns="sum").to_string(), "\n")
audios[["id", "fuente", "condicion", "licencia", "duracion_seg"]]

Clips: 29 | minutos: 53.0

                   count  minutos
fuente                           
OpenSLR 73             6      0.7
Wikimedia Commons     23     52.4 



,id,fuente,condicion,licencia,duracion_seg
0,clip_001,Wikimedia Commons,calle,CC BY-SA 4.0,136.3
1,clip_002,Wikimedia Commons,"calle, 2 hablantes",CC BY-SA 4.0,189.0
2,clip_003,Wikimedia Commons,calle,CC BY-SA 4.0,20.2
3,clip_004,Wikimedia Commons,conferencia de prensa,CC BY-SA 4.0,250.6
4,clip_005,Wikimedia Commons,fiesta patronal,CC BY-SA 4.0,98.3
5,clip_006,Wikimedia Commons,fiesta patronal,CC BY-SA 4.0,81.6
6,clip_007,Wikimedia Commons,fiesta patronal,CC BY-SA 4.0,39.2
7,clip_008,Wikimedia Commons,calle,CC BY-SA 4.0,161.1
8,clip_009,Wikimedia Commons,calle,CC BY-SA 4.0,289.3
9,clip_010,Wikimedia Commons,calle,CC BY-SA 4.0,207.8


## 4 · ASR: transcripción, diarización y exportación

Usamos `voz.py` de la Sesión 13 con esta configuración, definida antes de ver ningún resultado:

- **Modelo:** faster-whisper *large-v3-turbo* (CTranslate2, int8) en CPU, con `voz.Transcriptor("faster-whisper")` y `beam_size=5`.
  La línea base *small* que pide la Parte 2 se corre en `02_evaluacion.ipynb`.
- **VAD activado** (Silero, `vad_filter=True`, pausas de 0,3 s): Whisper solo recibe los tramos con voz, lo que evita que "alucine" frases en silencios, ruido o música.
- **Regla de respaldo del VAD** (se aplica igual a todos los clips): si el VAD reconoce voz en menos del 30 % del clip, ese clip se transcribe
  completo sin VAD y se marca para revisión manual de alucinaciones. Se agregó al ver que, en dos clips con audio muy fuerte y ruidoso
  (fiesta patronal y feria), el VAD casi no detectaba voz aunque al escucharlos sí hay una entrevista completa.
- **Idioma:** primero se **detecta** con los 30 s iniciales de cada clip (columna `idioma` y su probabilidad). Si un clip no sale en español, se reporta como descarte.
  Luego se transcribe con el idioma fijado en `es` (español; la variante peruana está documentada por cada fuente), para que el ruido no provoque cambios de idioma a mitad del clip.
- **Diarización en todos los clips** con `voz.diarizar` (huellas de voz WeSpeaker ResNet34 + agrupamiento jerárquico, umbral coseno 0,7, sin fijar el número de hablantes).
  De ahí sale `n_hablantes`. Cuando hay más de una voz, `texto_asr` lleva marcas `[HABLANTE_1]`, `[HABLANTE_2]`…
- **Exportación** por clip a `datos/raw/audio/asr/`: tabla de segmentos (`.csv`), subtítulos (`.srt`) y texto (`.txt`).
  Si un clip ya fue transcrito, se reutiliza su respaldo.


In [10]:
ASR_DIR = RAW_AUDIO / "asr"
ASR_DIR.mkdir(exist_ok=True)

transcriptor = voz.Transcriptor("faster-whisper")
MODELO_ASR = "faster-whisper large-v3-turbo (int8, CPU, beam 5, VAD on, idioma es)"
VOZ_MINIMA = 0.30          # fracción mínima del clip que el VAD debe reconocer como voz
print(transcriptor, "| carga:", round(transcriptor.segundos_carga, 1), "s")


def con_marcas(segmentos):
    "Une los segmentos en un texto; si hay varias voces, marca cada cambio de hablante."
    if len({s.hablante for s in segmentos}) <= 1:
        return " ".join(s.texto for s in segmentos)
    bloques, actual = [], None
    for s in segmentos:
        if s.hablante != actual:
            actual = s.hablante
            bloques.append(f"[HABLANTE_{actual[1:]}] {s.texto}")
        else:
            bloques[-1] += " " + s.texto
    return " ".join(bloques)


def transcribir_clip(id_, ruta):
    "Detecta idioma, transcribe con VAD, diariza y exporta CSV/SRT/TXT. Reutiliza el respaldo si existe."
    meta_json = ASR_DIR / f"{id_}.json"
    if meta_json.exists():
        meta = json.loads(meta_json.read_text(encoding="utf-8"))
        tabla = pd.read_csv(ASR_DIR / f"{id_}.csv").fillna({"texto": ""})
        segmentos = [voz.Segmento(r.inicio, r.fin, r.texto, r.hablante) for r in tabla.itertuples()]
        return meta, segmentos
    audio, _ = sf.read(ruta, dtype="float32")
    _, info = transcriptor.modelo.transcribe(audio[: 30 * voz.SR], language=None)   # solo detecta el idioma
    t0 = time.perf_counter()
    segmentos = transcriptor.segmentos(audio, idioma="es")
    vad = "on"
    if sum(s.fin - s.inicio for s in segmentos) < VOZ_MINIMA * len(audio) / voz.SR:
        # el VAD falla con audio muy fuerte y ruidoso: transcribir el clip completo y revisarlo a mano
        segs, _ = transcriptor.modelo.transcribe(audio, language="es", beam_size=5, vad_filter=False)
        segmentos = [voz.Segmento(round(s.start, 2), round(s.end, 2), s.text.strip()) for s in segs if s.text.strip()]
        vad = "off (respaldo: el VAD detectó voz en menos del 30 % del clip)"
    segundos_asr = time.perf_counter() - t0
    t0 = time.perf_counter()
    segmentos = voz.diarizar(audio, segmentos) if segmentos else []
    segundos_diar = time.perf_counter() - t0
    voz.a_tabla(segmentos).to_csv(ASR_DIR / f"{id_}.csv", index=False)
    (ASR_DIR / f"{id_}.srt").write_text(voz.a_srt(segmentos), encoding="utf-8")
    (ASR_DIR / f"{id_}.txt").write_text(voz.a_texto(segmentos), encoding="utf-8")
    meta = {"id": id_, "idioma_detectado": info.language, "prob_idioma": round(info.language_probability, 3),
            "duracion_seg": round(len(audio) / voz.SR, 1), "segundos_asr": round(segundos_asr, 1),
            "segundos_diarizacion": round(segundos_diar, 1), "vad": vad, "modelo_asr": MODELO_ASR}
    meta_json.write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    return meta, segmentos


asr = {}
for id_, ruta in zip(audios["id"], audios["archivo"]):
    asr[id_] = transcribir_clip(id_, ruta)
    m = asr[id_][0]
    print(f"{id_}: {m['duracion_seg']:6.1f} s de audio -> ASR {m['segundos_asr']:5.1f} s | idioma {m['idioma_detectado']} ({m['prob_idioma']})")

Transcriptor(faster-whisper (Whisper turbo en CTranslate2) en cpu) | carga: 1.6 s
clip_001:  136.3 s de audio -> ASR  23.8 s | idioma es (0.999)
clip_002:  189.0 s de audio -> ASR  36.1 s | idioma es (0.999)
clip_003:   20.2 s de audio -> ASR   4.0 s | idioma es (0.935)
clip_004:  250.6 s de audio -> ASR  54.1 s | idioma es (0.997)


clip_005:   98.3 s de audio -> ASR  33.4 s | idioma es (0.951)
clip_006:   81.6 s de audio -> ASR  21.5 s | idioma es (0.977)
clip_007:   39.2 s de audio -> ASR   9.8 s | idioma es (0.999)
clip_008:  161.1 s de audio -> ASR  42.7 s | idioma es (0.999)
clip_009:  289.3 s de audio -> ASR  83.1 s | idioma es (0.998)
clip_010:  207.8 s de audio -> ASR  59.3 s | idioma es (0.989)
clip_011:  110.1 s de audio -> ASR  37.0 s | idioma es (0.977)
clip_012:  149.7 s de audio -> ASR  44.5 s | idioma es (0.999)
clip_013:  181.2 s de audio -> ASR  49.1 s | idioma es (0.999)


clip_014:   90.1 s de audio -> ASR  17.9 s | idioma es (0.988)
clip_015:   89.1 s de audio -> ASR  21.5 s | idioma es (0.986)
clip_016:   99.0 s de audio -> ASR  37.9 s | idioma es (0.998)
clip_017:   99.6 s de audio -> ASR  16.7 s | idioma es (0.959)
clip_018:  104.2 s de audio -> ASR  33.8 s | idioma es (0.911)
clip_019:  187.1 s de audio -> ASR  51.8 s | idioma es (0.999)
clip_020:  103.4 s de audio -> ASR  36.9 s | idioma es (0.995)
clip_021:  196.0 s de audio -> ASR  66.6 s | idioma es (0.999)
clip_022:   96.1 s de audio -> ASR  28.5 s | idioma es (0.999)
clip_023:  163.9 s de audio -> ASR  51.0 s | idioma es (0.999)
clip_024:    6.9 s de audio -> ASR   5.7 s | idioma es (1.0)
clip_025:    5.0 s de audio -> ASR   5.5 s | idioma es (0.999)
clip_026:    8.5 s de audio -> ASR   5.7 s | idioma es (0.999)
clip_027:    5.6 s de audio -> ASR   5.5 s | idioma es (0.999)
clip_028:    5.4 s de audio -> ASR   5.5 s | idioma es (0.999)
clip_029:    8.0 s de audio -> ASR   6.6 s | idioma es (1

In [11]:
ALUCINACIONES = ["subtítulos", "subtitulos", "gracias por ver", "suscríbete", "amara.org"]


def revisar(id_):
    meta, segmentos = asr[id_]
    texto = " ".join(s.texto for s in segmentos)
    palabras = len(texto.split())
    voz_seg = sum(s.fin - s.inicio for s in segmentos)
    repetidos = pd.Series([s.texto.lower() for s in segmentos if len(s.texto.split()) >= 3]).value_counts()
    alertas = [] if meta.get("vad", "on") == "on" else ["sin VAD: revisar a mano"]
    if palabras == 0:
        alertas.append("sin voz")
    if meta["idioma_detectado"] != "es":
        alertas.append(f"idioma {meta['idioma_detectado']}")
    if voz_seg and palabras / (voz_seg / 60) > 220:
        alertas.append("demasiadas palabras por minuto")
    if (repetidos >= 3).any():
        alertas.append("frase repetida 3+ veces")
    if any(a in texto.lower() for a in ALUCINACIONES):
        alertas.append("frase típica de alucinación")
    return {"id": id_, "idioma": meta["idioma_detectado"], "prob_idioma": meta["prob_idioma"],
            "duracion_seg": meta["duracion_seg"], "segundos_voz": round(voz_seg, 1),
            "palabras": palabras, "palabras_por_min_voz": round(palabras / (voz_seg / 60)) if voz_seg else 0,
            "n_hablantes": len({s.hablante for s in segmentos}), "segundos_asr": meta["segundos_asr"],
            "factor_tiempo_real": round(meta["segundos_asr"] / meta["duracion_seg"], 3),
            "alertas": ", ".join(alertas), "texto_asr": con_marcas(segmentos)}


asr_clips = audios[["id", "fuente", "condicion"]].merge(pd.DataFrame([revisar(i) for i in audios["id"]]), on="id")
total = asr_clips["segundos_asr"].sum()
print(f"Audio: {asr_clips['duracion_seg'].sum() / 60:.1f} min | tiempo de ASR: {total / 60:.1f} min "
      f"| factor de tiempo real: {total / asr_clips['duracion_seg'].sum():.3f}")
print("Clips con 2+ hablantes:", (asr_clips["n_hablantes"] > 1).sum(), "| clips con alertas:", (asr_clips["alertas"] != "").sum(), "\n")
asr_clips.drop(columns="texto_asr")

# Descartes confirmados al escuchar el audio (se reportan y se excluyen de transcripciones.csv)
DESCARTES_AUDIO = {
    "clip_017": "Audio sin voz inteligible: al escucharlo no se entiende la mayor parte; Whisper alucina "
                "('Gracias por ver el video') o repite una misma frase, con y sin VAD.",
}
asr_clips["descartado"] = asr_clips["id"].isin(DESCARTES_AUDIO)
validos = asr_clips[~asr_clips["descartado"]]
print("Descartes:", len(DESCARTES_AUDIO), DESCARTES_AUDIO)
print(f"Corpus de audio válido: {len(validos)} clips | {validos['duracion_seg'].sum() / 60:.1f} min | "
      f"clips con 2+ hablantes: {(validos['n_hablantes'] > 1).sum()}")

Audio: 53.0 min | tiempo de ASR: 14.9 min | factor de tiempo real: 0.281
Clips con 2+ hablantes: 20 | clips con alertas: 3 

Descartes: 1 {'clip_017': "Audio sin voz inteligible: al escucharlo no se entiende la mayor parte; Whisper alucina ('Gracias por ver el video') o repite una misma frase, con y sin VAD."}
Corpus de audio válido: 28 clips | 51.4 min | clips con 2+ hablantes: 20


El clip con dos hablantes que usamos para comprobar la diarización (`clip_002`, dos vendedoras en El Agustino):

In [12]:
voz.a_tabla(asr["clip_002"][1]).head(12)

,inicio,fin,hablante,texto
0,0.02,8.02,H1,"Buenos días profesores y compañeros, hoy estoy cerca de aquí por mi casa y v..."
1,8.72,10.02,H1,Estoy con las señoras...
2,10.62,12.84,H2,"Isabel Vázquez, soy Xabarrientos."
3,13.40,19.86,H1,"Muy bien, voy a comenzar con la primera pregunta y es, ¿qué expectativas tie..."
4,21.96,30.36,H3,Que el afición se encuentra motivado para ganar este partido.
5,30.36,31.90,H1,¿Me puede decir su nombre?
6,32.20,33.24,H3,Isabel Vázquez.
7,33.95,47.59,H2,"La expectativa que yo pienso es que acá, que el Perú va a dar todo por todo,..."
8,48.46,48.92,H1,Al mundial.
9,49.08,49.48,H2,Al mundial.
